# Deliverable 3 Extended Evaluation

This notebook documents the Deliverable 3 refinement work. It focuses on:

- calibrated operating modes (`balanced` and `strict`)
- updated evaluation artifacts generated from the saved experiment CSVs
- a comparison between the original `0.50` threshold and the refined Deliverable 3 settings
- source-level stress-test analysis for responsible-AI reflection


In [ ]:
from pathlib import Path
import csv
import json

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
summary_path = repo_root / 'results' / 'deliverable3' / 'deliverable3_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
summary['production_recommendation']


## Operating Modes

Deliverable 2 effectively used the baseline classifier with a fixed `0.50` threshold. In Deliverable 3,
the system keeps that baseline model but adds two explicit operating modes:

- `balanced`: threshold `0.51`, chosen because it slightly improves clean-test F1
- `strict`: threshold `0.85`, chosen because it meaningfully reduces false positives on wrapper-heavy mutated prompts

The saved adversarial/cache-backed run is kept as a research comparison, not as the final deployment default.


In [ ]:
operating_modes_path = repo_root / 'results' / 'deliverable3' / 'operating_modes.csv'
rows = list(csv.DictReader(operating_modes_path.open(newline='', encoding='utf-8')))
for row in rows:
    print(row['mode_name'], row['dataset_variant'], row['threshold'], row['accuracy'], row['precision'], row['recall'], row['f1'])


## Threshold Sweep

The threshold sweep artifact makes the refinement reproducible without rerunning model training. If `matplotlib`
is installed in the environment, the following cell plots the clean-test and mutated-data F1 values for Model A.


In [ ]:
threshold_sweep_path = repo_root / 'results' / 'deliverable3' / 'threshold_sweep.csv'
sweep_rows = list(csv.DictReader(threshold_sweep_path.open(newline='', encoding='utf-8')))

model_a_rows = [row for row in sweep_rows if row['model_variant'] == 'Model A']
thresholds = sorted({float(row['threshold']) for row in model_a_rows})
clean = {float(row['threshold']): float(row['f1']) for row in model_a_rows if row['dataset_variant'] == 'test'}
mutated = {float(row['threshold']): float(row['f1']) for row in model_a_rows if row['dataset_variant'] == 'mutated_data'}

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 4.5))
    plt.plot(thresholds, [clean[t] for t in thresholds], label='Model A on test')
    plt.plot(thresholds, [mutated[t] for t in thresholds], label='Model A on mutated_data')
    plt.axvline(0.51, linestyle='--', label='Balanced threshold')
    plt.axvline(0.85, linestyle='--', label='Strict threshold')
    plt.xlabel('Decision threshold')
    plt.ylabel('F1 score')
    plt.title('Deliverable 3 threshold calibration for Model A')
    plt.legend()
    plt.tight_layout()
    plt.show()
except ModuleNotFoundError:
    print('matplotlib is not installed in this environment. The CSV artifact is still available at:')
    print(threshold_sweep_path)


## Subgroup Review

The Deliverable 3 analysis also exports `source_breakdown.csv` and `category_breakdown.csv`. Those files are useful for the
Responsible AI section of the report because they show where errors cluster instead of only reporting aggregate metrics.


In [ ]:
source_breakdown_path = repo_root / 'results' / 'deliverable3' / 'source_breakdown.csv'
source_rows = list(csv.DictReader(source_breakdown_path.open(newline='', encoding='utf-8')))

focus_rows = [
    row for row in source_rows
    if row['dataset_variant'] == 'mutated_data' and row['mode_name'] in {'balanced', 'strict'}
]
focus_rows = sorted(focus_rows, key=lambda row: (row['mode_name'], float(row['accuracy'])))
for row in focus_rows[:8]:
    print(row['mode_name'], row['group'], 'accuracy=', row['accuracy'], 'fp=', row['fp'], 'fn=', row['fn'])
